# Reference-state comparison

FleXgeo2 can compare every model in an ensemble to a reference state. The distance is Euclidean distance in each residue's `(curvature, torsion)` space, so large values mark residues whose local backbone geometry differs strongly from the reference.

In [ ]:
from pathlib import Path
import sys
import tempfile


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "pdb2lj5.pdb").is_file():
            return candidate
    raise FileNotFoundError("Could not find the FleXgeo2 repo root containing pyproject.toml and pdb2lj5.pdb")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import matplotlib.image as mpimg
    import matplotlib.pyplot as plt
    import melodia_py  # noqa: F401
    import pandas as pd
    from flexgeo2 import AnalysisConfig, FlexGeo2App, OutputConfig, ReferenceConfig
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Missing dependency: {exc.name}. Install the project in your notebook kernel with `pip install -e .` "
        "or, if you use uv, run `uv sync` and select the project environment as the Jupyter kernel."
    ) from exc

PDB_FILE = REPO_ROOT / "pdb2lj5.pdb"
tmpdir = tempfile.TemporaryDirectory(prefix="flexgeo2_reference_")
OUTPUT_DIR = Path(tmpdir.name)
print(f"Temporary output directory: {OUTPUT_DIR}")

## Compare the ensemble to model 1

`ReferenceConfig(model_id="1")` selects model 1 from the input PDB as the reference state.

In [ ]:
result = FlexGeo2App().run(
    AnalysisConfig(
        pdb_file=PDB_FILE,
        chains=["A"],
        n_jobs=1,
        reference=ReferenceConfig(model_id="1"),
        output=OutputConfig(output_dir=OUTPUT_DIR, verbose=True, write_files=True),
    )
)

distance_result = result.distance_result
assert distance_result is not None
distance_result.reference_label

## Inspect distance tables

`long_df` has one row per model and residue. `summary_df` collapses the distances across the ensemble by residue.

In [ ]:
distance_result.long_df.head()

In [ ]:
distance_result.summary_df.head()

## Top residues by distance to reference

Mean distance highlights consistently different residues. Max distance highlights residues with at least one strongly divergent model.

In [ ]:
distance_result.summary_df.sort_values("distance_mean", ascending=False).head(10)[
    ["chain", "residue_label", "distance_mean", "distance_std", "distance_max", "models"]
]

In [ ]:
distance_result.summary_df.sort_values("distance_max", ascending=False).head(10)[
    ["chain", "residue_label", "distance_mean", "distance_std", "distance_max", "models"]
]

## Distance heatmap

The heatmap puts models on one axis and residues on the other. Brighter regions are farther from the reference model in curvature/torsion space.

In [ ]:
assert result.outputs.distance_heatmap is not None and result.outputs.distance_heatmap.is_file()
img = mpimg.imread(result.outputs.distance_heatmap)
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.axis("off")
plt.show()

## External references

To compare against a different PDB file, use `ReferenceConfig(pdb_file="path/to/reference.pdb", pdb_model_id="1")`. If `pdb_model_id` is omitted, FleXgeo2 uses the first model in the reference file.